# Credit Scoring with Fairness Improvement

This machine learning project's goal is to:
- Build a credit scoring model
- Observe bias between gender groups
- Apply mitigation technique
- Improve fairness and compare results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Load the Dataset

We use German credit dataset where:
- Each row represents one person
- Columns contain personal and financial details
- The target is credit risk (good or bad)

In [ ]:
df = pd.read_csv("/kaggle/input/german-credit-data-with-risk/german_credit_data.csv")
df.head()

In [ ]:
df.info()

## Data Cleaning

We fill missing values with a simple placeholder.

In [ ]:
df.fillna("Unknown", inplace=True)

## Convert Target Column

Machine learning models work with numbers.
We convert:
- good → 1
- bad → 0

In [ ]:
df['Risk'] = df['Risk'].map({'good': 1, 'bad': 0})

## Convert Categorical Data

We convert text columns (like gender) into numbers
using one-hot encoding.

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.head()

## Separate Features and Target

- X → input data
- y → output (credit risk)

In [ ]:
X = df_encoded.drop('Risk', axis=1)
y = df_encoded['Risk']

## Train–Test Split

The model learns from training data
and is tested on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train the Model

We use Logistic Regression because it is:
- Simple
- Easy to explain
- Suitable for binary classification

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

## Evaluate Accuracy

Accuracy tells us how often the model predicts correctly.

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
accuracy

## Fairness Check (Before Mitigation)

We check whether approval rates differ
between male and female groups.


In [ ]:
test_df = df_encoded.loc[y_test.index].copy()
test_df['prediction'] = y_pred

male_rate_before = test_df[test_df['Sex_male'] == 1]['prediction'].mean()
female_rate_before = test_df[test_df['Sex_male'] == 0]['prediction'].mean()

male_rate_before, female_rate_before


## Bias Mitigation Technique

To reduce bias, we use a simple post-processing technique.
We adjust decision thresholds using prediction probabilities.


In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
test_df['probability'] = y_prob
test_df['prediction_default'] = (test_df['probability'] >= 0.5).astype(int)

# Apply Mitigation

test_df['prediction_mitigated'] = test_df['prediction_default']

test_df.loc[
    (test_df['Sex_male'] == 0) & (test_df['probability'] >= 0.45),
    'prediction_mitigated'
] = 1


## Fairness Check (After Mitigation)

In [ ]:
male_rate_after = test_df[test_df['Sex_male'] == 1]['prediction_mitigated'].mean()
female_rate_after = test_df[test_df['Sex_male'] == 0]['prediction_mitigated'].mean()

male_rate_after, female_rate_after

## Fairness Comparison Plots


In [ ]:
plt.figure()
plt.bar(['Male', 'Female'], [male_rate_before, female_rate_before])
plt.title("Approval Rate by Gender (Before Mitigation)")
plt.ylabel("Approval Rate")
plt.ylim(0, 1)
plt.show()


In [ ]:
plt.figure()
plt.bar(['Male', 'Female'], [male_rate_after, female_rate_after])
plt.title("Approval Rate by Gender (After Mitigation)")
plt.ylabel("Approval Rate")
plt.ylim(0, 1)
plt.show()


## Conclusion

This project demonstrates how bias can exist
in machine learning models used for credit scoring.

By applying a simple threshold-based mitigation technique,
we were able to reduce the difference in approval rates
between male and female groups.

This shows that fairness can be improved even with
simple models and simple techniques.
